In [1]:
import pandas as pd, numpy as np
df = pd.read_csv("train.csv", parse_dates=["pickup_datetime", "dropoff_datetime"])
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe())

(1458644, 11)
id                            object
vendor_id                      int64
pickup_datetime       datetime64[ns]
dropoff_datetime      datetime64[ns]
passenger_count                int64
pickup_longitude             float64
pickup_latitude              float64
dropoff_longitude            float64
dropoff_latitude             float64
store_and_fwd_flag            object
trip_duration                  int64
dtype: object
id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64
          vendor_id                pickup_datetime  \
count  1.458644e+06                        1458644   
mean   1.534950e+00  2016-04-01 10:10:24.940037120   
min    1.000000e+00            2016-01-01 00:00:17   
25%    1.000000e+00  2016-02-17 16:46:04.249999872   
50%    2.000000e+0

In [2]:
raw = pd.read_csv("train.csv", parse_dates=["pickup_datetime", "dropoff_datetime"])
rules = {
    "duration < 10s or > 4h": (raw.trip_duration < 10) | (raw.trip_duration > 14400),
    "coordinates outside NYC": ~(
        raw.pickup_longitude.between(-74.3, -73.7) & raw.dropoff_longitude.between(-74.3, -73.7) &
        raw.pickup_latitude.between(40.5, 41.0) & raw.dropoff_latitude.between(40.5, 41.0)),
    "passenger_count == 0 or > 6": ~raw.passenger_count.between(1, 6),
}
for name, m in rules.items():
    print(f"{name}: {m.sum()} rows")
bad = np.logical_or.reduce(list(rules.values()))
print("total removed:", bad.sum(), f"({bad.mean():.2%})", "| remaining:", (~bad).sum())

duration < 10s or > 4h: 4061 rows
coordinates outside NYC: 998 rows
passenger_count == 0 or > 6: 65 rows
total removed: 5061 (0.35%) | remaining: 1453583


In [3]:
# Data quality check: why the upper bound was tightened from 24h to 4h
df_v1 = pd.read_parquet("data_clean/train_clean.parquet")   # v1: 24h upper bound
long = df_v1[df_v1.trip_duration > 4 * 3600]
print(len(long), (df_v1.trip_duration > 20 * 3600).sum())

def dist_km(d):   # rough straight-line distance, only for judging orders of magnitude
    x = (d.dropoff_longitude - d.pickup_longitude) * 111.32 * np.cos(np.deg2rad(40.75))
    y = (d.dropoff_latitude - d.pickup_latitude) * 110.57
    return np.hypot(x, y)

print(dist_km(long).describe())

bins = [0, 1, 2, 3, 4, 6, 12, 20, 24]
print(pd.cut(df_v1.trip_duration / 3600, bins).value_counts().sort_index())

2067 1926
count    2067.000000
mean        4.143050
std         4.763787
min         0.000000
25%         1.327041
50%         2.390965
75%         4.610352
max        31.692265
dtype: float64
trip_duration
(0, 1]      1443488
(1, 2]         9925
(2, 3]          136
(3, 4]           34
(4, 6]           15
(6, 12]          66
(12, 20]         60
(20, 24]       1926
Name: count, dtype: int64
